# M5 Drift Experiment — Phases 3-6

Walk forward through 2013-2015, fire drift-triggered retrains per arm (`frozen`, `periodic`, `ewc`, `replay`, `sdft`), then compute the grid-anchored metrics, probe-based forgetting, and the accuracy-vs-#retrains tradeoff.

> Requires the base model from `m5_base_train.ipynb` (`outputs/drift/checkpoints/base/`).

## 0. Setup

In [ ]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings('ignore')
PROJECT_DIR = str(Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd())
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)
import drift_pipeline.core_pipeline as cp
from drift_pipeline import monitor as mon, retrain as rt, metrics as mt, plots as pl
print('project:', PROJECT_DIR, '| device:', cp.DEVICE)

## 1. Data + base model

In [ ]:
data = cp.prepare_drift_data()
cp.print_timeline_summary(data)
base = mon.load_base()   # restores arch-matched TFT + PPO + calibration

## 2. (Optional) smoke knobs — shrink for a fast end-to-end check

In [ ]:
SMOKE = False   # True = 2 arms, tiny retrains, for a quick wiring check
ARMS = None     # None -> frozen, periodic, ewc, replay, sdft
if SMOKE:
    cp.CONFIG['retrain']['retrain_epochs'] = 1
    cp.CONFIG['retrain']['retrain_timesteps'] = 800
    ARMS = ['frozen', 'sdft']
    print('SMOKE on -> arms:', ARMS)
else:
    print('FULL run -> all arms')

## 3. Run all arms (Phase 3+4) + metrics (Phase 5)

In [ ]:
out = mt.run_drift_experiment(data, arms=ARMS)
eff = out['metrics']['efficiency']
eff

## 4. Plots (Phase 6)

In [ ]:
figs = pl.generate_all_plots()
figs

In [ ]:
from IPython.display import Image, display
for k in ['accuracy_vs_retrains', 'error_timeline', 'profit_timeline',
          'retrain_counts', 'forgetting']:
    if k in figs:
        print(k); display(Image(figs[k]))

---
**Outputs:** `outputs/drift/results/metrics_*.csv` (accuracy / forgetting / efficiency) and `outputs/drift/plots/*.png`. The efficiency table + `accuracy_vs_retrains.png` are the headline result.